# 距離変換 boundary loss パイロット — fold1、baseアーキテクチャ固定

WINNING_STRATEGY_2026-08-29.md §6.12優先度4。region損失(0.35·BCE + 0.65·Dice)
は不変で、**境界項だけ**を morphological-gradient L1 から距離変換ベース
(`mean(sigmoid·φ_G)`、Kervadec系)に置き換える。φ_G は GT の符号付き距離写像
(内部負・外部正の離散距離近似、サンプルごとに±1正規化)。

| 構成 | 内容 |
|---|---|
| `baseline` | 現行 `refiner_loss` 完全同一(V30から不変) |
| `dtb` | 境界項だけ `0.15·mean(sigmoid·φ_G)` に置換(重み0.15はedge項から引継ぎ) |

fold1基準(標準refiner、matched-parameter ablation実測、2seed平均): 0.4405。
旧 baseline 再利用は既定 OFF。入力契約・fold・実 checkpoint の一致が確認できなければ
baseline も未実行として残す。これは CPU で構造検証した pilot であり、新しい GPU 受入結果ではない。


In [ ]:
# [1/8] Drive接続、設定、前提の確認
!pip -q install ultralytics pycocotools
import hashlib, json, math, os, random, shutil, sys, time
from datetime import datetime, timezone
from importlib.metadata import version
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from google.colab import drive
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

drive.mount("/content/drive")

class CFG:
    DRIVE_ROOT = Path("/content/drive/MyDrive/solar_filament_v30")
    DATA_CACHE = DRIVE_ROOT / "competition_data"
    ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
    PREPROC_CACHE = DRIVE_ROOT / "preprocessed_v30_q96"
    LOCAL_ROOT = Path("/content/solar_v59_bdt")
    PREPROC_DIR = LOCAL_ROOT / "preprocessed"
    # --- build_v51_edgeattnet_refiner.py / build_matched_param_ablation.py / v54 と同一。触らないこと ---
    CROP_SIZE = 320
    CROP_CONTEXT = 1.65
    REFINER_EPOCHS = 14
    REFINER_LR = 2e-4
    REFINER_BATCH_CANDIDATES = (8, 6, 4, 2)
    FOLD = 1
    PROTOCOL_SHA256 = 'bc9dc187d5c5b6cf8132e210137d88df7d4304455e32a56bf02d6b89900edbfd'
    REUSE_V54_BASELINE = False  # Legacy v54 has no equivalence proof; baseline remains pending.
    CONF = 0.26
    REFINER_THRESHOLD = 0.45
    DROP_THRESHOLD = 0.90
    MIN_AREA = 20
    BASELINE_PQ = 0.4405
    BDT_WEIGHT = 0.15
    SEEDS = (3407, 8407)
    CONFIGS = ('baseline', 'dtb')  # ("baseline", "dtb")
    V54_MANIFEST = DRIVE_ROOT / "artifacts" / "v54_ema_pilot_fold1_manifest.json"
    MANIFEST = DRIVE_ROOT / "artifacts" / "v59_boundary_dt_fold1_manifest.json"

CFG.LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
CFG.PREPROC_DIR.mkdir(parents=True, exist_ok=True)

required = {
    "competition data": CFG.DATA_CACHE,
    "grouped_folds.csv": CFG.ARTIFACT_ROOT / "grouped_folds.csv",
    "前処理画像": CFG.PREPROC_CACHE,
    f"fold{CFG.FOLD} YOLO(OOF推論用)": CFG.DRIVE_ROOT / "runs" / f"yolo_fold_{CFG.FOLD}" / "weights" / "best.pt",
}
missing = {k: str(v) for k, v in required.items() if not v.exists()}
if missing:
    raise FileNotFoundError(f"前提artifactが見つかりません: {missing}")
assert torch.cuda.is_available(), "GPUランタイムを選択してください"
print({k: "OK" for k in required})
print({"GPU": torch.cuda.get_device_name(0)})


In [ ]:
# [2/8] boundary_dt (solar_filament/pipeline/boundary_dt.py と同一、テスト済み)
"""Distance-transform boundary loss — WINNING_STRATEGY_2026-08-29.md §6.12 優先度4。

現行 `refiner_loss` の境界項は morphological-gradient(dilate−erode)の L1 で、
V30 から不変の粗い定式化(README.md §2.1.12)。このモジュールはその代替として
Kervadec らの距離変換ベース境界損失(boundary loss)を提供する:

    L_boundary = mean( sigmoid(logits) * phi_G )

ここで phi_G は GT マスクの正規化済み符号付き距離写像で、GT 内部で負、
外部で正となる離散画素上の近似。損失を下げる方向は「内部で p→1、外部で p→0」で、
勾配の大きさが境界からの距離に比例する = 境界から遠い誤りほど強く罰する。

計算は numpy/cv2 のみ(GPU不要)なのでローカルでテスト可能
(`tests/test_boundary_dt.py`)。notebook 側(`build_v59_boundary_dt_pilot.py`
が生成)ではバッチ内各サンプルの augmented target から phi を作り、
既存の region 損失(0.35*BCE + 0.65*Dice)に `0.15 * boundary` として足す
= 既存 edge 項の重み0.15をそのまま引き継ぐ単変数置換。
"""

from __future__ import annotations

import cv2
import numpy as np


def signed_distance_map(mask):
    """二値マスクの正規化済み符号付き距離写像 phi_G を返す。

    Args:
        mask: (H, W) の二値マスク。有限な 0/1 を受け付ける。

    Returns:
        float32 の (H, W) 配列。GT 内部で負(最深部で最小)、外部で正
        (最遠部で最大)。離散画素の境界近傍では小さい非ゼロ値を持つ。
        絶対値の最大が 1 になるよう正規化する。
        空マスク・全面マスク(境界が存在しない)は全ゼロを返す
        (境界信号が定義できないため、損失への寄与をゼロにする)。
    """
    array = np.asarray(mask)
    if not np.isfinite(array).all() or not np.isin(array, [0, 1]).all():
        raise ValueError("mask must contain finite binary values")
    binary = array.astype(np.uint8)
    if binary.ndim != 2:
        raise ValueError(f"mask must be 2-D, got shape {binary.shape}")
    area = int(binary.sum())
    if area == 0 or area == binary.size:
        return np.zeros(binary.shape, dtype=np.float32)
    # cv2.distanceTransform は各非ゼロ画素について最近傍ゼロ画素への距離を返す。
    #   inside  = GT 内部で「境界までの距離」、外部で 0
    #   outside = GT 外部で「境界までの距離」、内部で 0
    inside = cv2.distanceTransform(binary, cv2.DIST_L2, 5)
    outside = cv2.distanceTransform(1 - binary, cv2.DIST_L2, 5)
    phi = outside - inside  # 内部で負、外部で正
    peak = float(np.abs(phi).max())
    if peak <= 0:
        return np.zeros(binary.shape, dtype=np.float32)
    return (phi / peak).astype(np.float32)


def batch_signed_distance_maps(targets):
    """(N, H, W) または (N, 1, H, W) のターゲット列から phi の列を作る。

    Returns:
        float32 の (N, H, W) 配列(targets が4次元なら (N, 1, H, W) の次元は落とす)。
    """
    array = np.asarray(targets)
    if array.ndim == 4:
        if array.shape[1] != 1:
            raise ValueError("targets must have exactly one channel")
        array = array[:, 0]
    if array.ndim != 3:
        raise ValueError(f"targets must be (N,H,W) or (N,1,H,W), got shape {array.shape}")
    return np.stack([signed_distance_map(t) for t in array])


def boundary_term(probabilities, phi):
    """mean(p * phi_G) — 距離変換境界損失の本体(numpy評価用・テスト用)。

    学習時は torch 側で `(logits.sigmoid() * phi).mean()` として同じ式を評価する。
    完全な予測(p=GT)なら負値(最小方向)、空予測なら 0、GT外を立てるほど正に増える。
    """
    p = np.asarray(probabilities, dtype=np.float64)
    g = np.asarray(phi, dtype=np.float64)
    if p.shape != g.shape:
        raise ValueError(f"shape mismatch: probabilities {p.shape} vs phi {g.shape}")
    if not p.size or not np.isfinite(p).all() or not np.isfinite(g).all():
        raise ValueError("probabilities and distances must be nonempty and finite")
    if (p < 0).any() or (p > 1).any():
        raise ValueError("probabilities must be in [0, 1]")
    return float((p * g).mean())


In [ ]:
# [3/8] COCOとfoldを読み、前処理画像をローカルSSDへ展開
def find_dataset_root(base):
    matches = list(Path(base).rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
    if not matches:
        raise FileNotFoundError(f"学習annotation JSONが見つかりません: {base}")
    return matches[0].parents[2]

TRAIN_JSON = next(find_dataset_root(CFG.DATA_CACHE).rglob(
    "MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
with open(TRAIN_JSON, encoding="utf-8") as handle:
    coco = json.load(handle)

images = {item["id"]: item for item in coco["images"]}
annotations = defaultdict(list)
for ann in coco["annotations"]:
    annotations[ann["image_id"]].append(ann)
file_to_ids = defaultdict(list)
for image_id, info in images.items():
    file_to_ids[info["file_name"]].append(image_id)
train_files = sorted(file_to_ids)

fold_frame = pd.read_csv(CFG.ARTIFACT_ROOT / "grouped_folds.csv")
if fold_frame["file_name"].duplicated().any() or not fold_frame["fold"].isin([0, 1, 2]).all():
    raise ValueError("Grouped folds must have unique files and integer folds 0/1/2")
fold_of_file = dict(zip(fold_frame["file_name"], fold_frame["fold"].astype(int)))
assert all(name in fold_of_file for name in train_files), "fold未割当の観測がある"

preprocessed_hashes = {}
for name in tqdm(train_files, desc="前処理画像をSSDへ・SHA256"):
    local = CFG.PREPROC_DIR / name
    local.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(CFG.PREPROC_CACHE / name, local)
    preprocessed_hashes[name] = hashlib.sha256(local.read_bytes()).hexdigest()

try:
    from pycocotools import mask as coco_mask
except ImportError:
    !pip -q install pycocotools
    from pycocotools import mask as coco_mask

def ann_mask(image_id, ann):
    info = images[image_id]
    height, width = int(info["height"]), int(info["width"])
    segmentation = ann["segmentation"]
    if isinstance(segmentation, list):
        rle = coco_mask.merge(coco_mask.frPyObjects(segmentation, height, width))
    elif isinstance(segmentation.get("counts"), list):
        rle = coco_mask.frPyObjects(segmentation, height, width)
    else:
        rle = segmentation
    return coco_mask.decode(rle).astype(np.uint8)

train_files_fold = [n for n in train_files if fold_of_file[n] != CFG.FOLD]
valid_files_fold = [n for n in train_files if fold_of_file[n] == CFG.FOLD]
assert not (set(train_files_fold) & set(valid_files_fold)), "fold分割が壊れている"
print({"観測": len(train_files), "除外train": len(train_files_fold),
       "検証(OOF)": len(valid_files_fold), "annotation": len(coco["annotations"])})
if not train_files_fold or not valid_files_fold:
    raise ValueError("Both training and validation observations are required")

def atomic_save(path, value, is_checkpoint=False):
    import os
    import tempfile
    path = Path(path)
    payload = None if is_checkpoint else json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False).encode("utf-8")
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(mode="wb", dir=path.parent, suffix=".tmp", delete=False) as handle:
            temporary = Path(handle.name)
            if is_checkpoint:
                torch.save(value, handle)
            else:
                handle.write(payload)
            handle.flush()
            os.fsync(handle.fileno())
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()

def validate_paired_rows(rows):
    pairs = {}
    for row in rows:
        pairs.setdefault(row["seed"], []).append(row)
    for arms in pairs.values():
        if len({r["batch_size"] for r in arms}) != 1 or len({r["initial_state_sha256"] for r in arms}) != 1:
            raise ValueError("Paired arms must share batch size and actual initial state")


def file_sha256(path):
    import hashlib
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def validate_checkpoint(checkpoint, config, seed):
    if (not isinstance(checkpoint, dict) or checkpoint.get("training_status") != "complete"
            or checkpoint.get("input_contract") != PILOT_CONTRACT
            or checkpoint.get("config") != config or checkpoint.get("seed") != seed
            or checkpoint.get("fold") != CFG.FOLD or not checkpoint.get("best_model")
            or not checkpoint.get("training_summary")):
        raise ValueError("Checkpoint is incomplete or differs from this pilot; retain it and use a fresh path")
    initial = checkpoint.get("initial_state_sha256")
    if not isinstance(initial, str) or len(initial) != 64 or any(c not in "0123456789abcdef" for c in initial):
        raise ValueError("Checkpoint has no actual initial-state fingerprint")
    return checkpoint

def validate_run_row(row):
    if (not isinstance(row, dict) or row.get("config") not in CFG.CONFIGS
            or row.get("seed") not in CFG.SEEDS or row.get("fold") != CFG.FOLD
            or row.get("input_contract") != PILOT_CONTRACT
            or row.get("checkpoint_selection") != "best_model"):
        raise ValueError("Run fold/arm/input contract does not match this pilot")
    for metric in ("pq", "sq", "rq"):
        value = row.get(metric)
        if type(value) not in (int, float) or not math.isfinite(value) or not 0 <= value <= 1:
            raise ValueError("Run has missing or invalid quality metrics")
    for metric in ("tp", "fp", "fn"):
        if type(row.get(metric)) is not int or row[metric] < 0:
            raise ValueError("Run has missing or invalid match counts")
    for field in ("checkpoint_sha256", "initial_state_sha256"):
        value = row.get(field)
        if not isinstance(value, str) or len(value) != 64 or any(c not in "0123456789abcdef" for c in value):
            raise ValueError("Run is missing checkpoint/initial-state fingerprints")
    checkpoint = Path(row.get("checkpoint", ""))
    if not checkpoint.is_file() or file_sha256(checkpoint) != row["checkpoint_sha256"]:
        raise ValueError("Recorded checkpoint is absent or its bytes changed")
    if type(row.get("batch_size")) is not int or row["batch_size"] <= 0:
        raise ValueError("Run has no valid training batch size")
    return row

def read_pilot_manifest():
    if not CFG.MANIFEST.exists():
        return {"experiment_id": f"v59_boundary_dt_fold{CFG.FOLD}", "input_contract": PILOT_CONTRACT, "runs": []}
    manifest = json.loads(CFG.MANIFEST.read_text(encoding="utf-8"))
    if (not isinstance(manifest, dict) or manifest.get("input_contract") != PILOT_CONTRACT
            or manifest.get("experiment_id") != f"v59_boundary_dt_fold{CFG.FOLD}"
            or not isinstance(manifest.get("runs"), list)):
        raise ValueError("Manifest contract differs or is absent; retain it and use a fresh path")
    seen = set()
    for row in manifest["runs"]:
        validate_run_row(row)
        key = (row["config"], row["seed"])
        if key in seen:
            raise ValueError("Duplicate pilot run")
        seen.add(key)
    validate_paired_rows(manifest["runs"])
    return manifest

def reuse_baselines(source):
    if not CFG.REUSE_V54_BASELINE:
        return []
    if not isinstance(source, dict) or source.get("input_contract") != PILOT_CONTRACT:
        raise ValueError("Baseline equivalence is unverified; rerun the paired baseline")
    result, seen = [], set()
    for row in source.get("runs", []):
        if row.get("config") != "baseline":
            continue
        validate_run_row(row)
        key = (row["config"], row["seed"])
        if key in seen:
            raise ValueError("Duplicate reused baseline")
        seen.add(key)
        result.append({**row, "reused_from": "explicit_contract_checked_baseline"})
    validate_paired_rows(result)
    return result

PILOT_CONTRACT = {
    "schema_version": 1, "fold": CFG.FOLD, "protocol_sha256": CFG.PROTOCOL_SHA256,
    "annotations_sha256": file_sha256(TRAIN_JSON),
    "folds_sha256": file_sha256(CFG.ARTIFACT_ROOT / "grouped_folds.csv"),
    "detector_sha256": file_sha256(CFG.DRIVE_ROOT / "runs" / f"yolo_fold_{CFG.FOLD}" / "weights/best.pt"),
    "preprocessed_sha256": hashlib.sha256(json.dumps(preprocessed_hashes, sort_keys=True).encode()).hexdigest(),
    "recipe": {name: getattr(CFG, name) for name in ("CROP_SIZE", "CROP_CONTEXT", "REFINER_EPOCHS", "REFINER_LR",
        "REFINER_BATCH_CANDIDATES", "CONF", "REFINER_THRESHOLD", "DROP_THRESHOLD", "MIN_AREA", "BDT_WEIGHT")},
    "seeds": list(CFG.SEEDS), "configs": list(CFG.CONFIGS),
    "runtime_versions": {"python": sys.version, "torch": torch.__version__, "numpy": np.__version__, "opencv": cv2.__version__, "ultralytics": version("ultralytics"), "pycocotools": version("pycocotools")},
}
PILOT_CONTRACT = json.loads(json.dumps(PILOT_CONTRACT, sort_keys=True, allow_nan=False))
read_pilot_manifest()  # Reject old/mixed results before model loading or historical writes.


In [ ]:
# [4/8] MediSegUNet (solar_filament/models/mediseg_models.py と同一。base構成のみ使用)
"""
MediSeg-AI Knowledge Port for Solar Filament Segmentation
Includes:
1. MediSeg UNet
2. MediSeg DeepLabV3Plus with ASPP (Atrous Spatial Pyramid Pooling)
3. CombinedLoss (BCE + Dice Loss)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

# ============================================================
# 1. Combined Loss (BCE + Dice Loss)
# ============================================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# ============================================================
# 2. MediSeg UNet Architecture
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MediSegUNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1, bilinear: bool = False):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

# ============================================================
# 3. MediSeg DeepLabV3Plus with ASPP Module
# ============================================================
class ASPP(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 256):
        super().__init__()
        self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1x1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=6, dilation=6, bias=False)
        self.bn3x3_1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_2 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=12, dilation=12, bias=False)
        self.bn3x3_2 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=18, dilation=18, bias=False)
        self.bn3x3_3 = nn.BatchNorm2d(out_channels)
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_global = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn_global = nn.BatchNorm2d(out_channels)
        
        self.conv_final = nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False)
        self.bn_final = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        x1 = self.relu(self.bn1x1(self.conv1x1(x)))
        x2 = self.relu(self.bn3x3_1(self.conv3x3_1(x)))
        x3 = self.relu(self.bn3x3_2(self.conv3x3_2(x)))
        x4 = self.relu(self.bn3x3_3(self.conv3x3_3(x)))
        
        x5 = self.global_avg_pool(x)
        x5 = self.relu(self.bn_global(self.conv_global(x5)))
        x5 = F.interpolate(x5, size=size, mode='bilinear', align_corners=True)
        
        x_cat = torch.cat([x1, x2, x3, x4, x5], dim=1)
        return self.relu(self.bn_final(self.conv_final(x_cat)))

class MediSegDeepLabV3Plus(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1):
        super().__init__()
        # Feature Extractor
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 512, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.aspp = ASPP(512, 256)
        self.classifier = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, out_channels, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[2:]
        feat = self.stem(x)
        aspp_feat = self.aspp(feat)
        out = self.classifier(aspp_feat)
        return F.interpolate(out, size=input_size, mode='bilinear', align_corners=True)

# ============================================================
# 4. Edge-Guided Attention U-Net (EG-MHSA)
# ============================================================
# 2026-08-29、WINNING_STRATEGY_2026-08-29.md §1.4のPhase 2実験用に追加。
# 参考文献: "EdgeAttNet: Towards Barb-Aware Filament Segmentation"
# (arXiv:2509.02964)。MAGFiLO上でU-Net(mIoU_pairwise 0.5724)から
# EdgeAttNet(0.6451)への改善を報告しており、barb(フィラメントの微細な
# 枝構造)の認識精度向上を狙ったアーキテクチャ。本プロジェクトの
# refiner(MediSegUNet)が抱える主要な弱点(near-miss境界品質)と直接
# 対応するため、置き換え候補として追試する。
#
# 【重要な注意】著者の公開リポジトリ(github.com/dasjar/EdgeAttNet)を
# 直接確認したところ、`model/edgeattnet_model.py`(本来UNetEdgeTransformer
# クラスが定義されているべきファイル)の中身が誤って`main.py`と同一に
# なっており、実際のアーキテクチャコードはリポジトリ上に存在しない
# (2026-08-29、`gh api`でリポジトリ全ファイルを走査して確認)。学習済み
# 重みの配布も確認できなかった。**以下は論文本文のアーキテクチャ記述
# (edge prior E=σ(Conv3x3(入力画像))をbottleneckへbilinear補間し
# 1x1convで射影、Q・Kへ加算、位置エンコーディングなし)のみを根拠にした
# 独自の再実装であり、著者オリジナル実装とのコード上の一致は未検証。**
# ベースにするU-Net本体はMediSegUNetをそのまま流用しているため、
# パラメータ総数は論文記載の値(22.6M)とは一致しない。


class EdgeGuidedMHSA(nn.Module):
    """Edge-Guided Multi-Head Self-Attention (EG-MHSA)。

    U-NetのbottleneckでSelf-Attentionを行う際、入力画像から抽出した
    エッジ事前分布をQuery/Keyに加算し、境界・barb構造への感度を高める。
    Value・出力チャネル数はbottleneckの特徴マップと同一に保つ
    (MediSegUNetへdrop-inできるよう設計)。
    """

    def __init__(self, in_channels: int, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        # E = sigmoid(Conv3x3(入力画像)): 単一チャンネルのエッジ事前分布
        self.edge_conv = nn.Conv2d(in_channels, 1, kernel_size=3, padding=1)
        # E' = Conv1x1(interpolate(E)): bottleneckのチャンネル数へ射影
        self.edge_proj = nn.Conv2d(1, bottleneck_channels, kernel_size=1)
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor, original_image: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        edge_map = torch.sigmoid(self.edge_conv(original_image))
        edge_map = F.interpolate(edge_map, size=(h, w), mode="bilinear", align_corners=True)
        edge_prior = self.edge_proj(edge_map)  # (B, C, H, W)

        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        edge_flat = edge_prior.flatten(2).transpose(1, 2)  # (B, N, C)
        qk = x_flat + edge_flat  # Q = K = x + E'。位置エンコーディングは使わない(論文通り)

        attn_out, _ = self.mhsa(qk, qk, x_flat)  # V = x(エッジ由来の加算なし)
        attn_out = self.norm(attn_out + x_flat)  # 残差接続 + LayerNorm(論文に明記はないが標準的なTransformerブロックとして採用)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegEdgeAttUNet(nn.Module):
    """MediSegUNetのbottleneckにEdgeGuidedMHSAを挿入したdrop-in代替。

    入出力インターフェースはMediSegUNetと完全に同一(in_channels=3の
    crop、out_channels=1の確率マップ)。既存のrefiner学習・推論コードを
    変更せず、モデルクラスの差し替えだけで比較できるように設計している。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.edge_attn = EdgeGuidedMHSA(in_channels, bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.edge_attn(x5, x)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


# ============================================================
# 5. Plain MHSA U-Net (matched-parameter ablation control, no edge prior)
# ============================================================
# 2026-08-31、report/main.tex Future Work item 1(matched-parameter
# ablation)のために追加。EdgeGuidedMHSAからedge_conv/edge_projを除いた
# だけの構成——Q=K=V=bottleneck特徴量の標準的なself-attentionで、
# edge priorをQ/Kへ加算する項だけがない。edge_conv/edge_projは
# 3(in_channels)*1*3*3+1 + 1*1024*1*1+1024 ≈ 2,076パラメータしかなく
# 全体(約35.2M)の0.006%程度なので、このクラスはMediSegEdgeAttUNetと
# ほぼ完全にパラメータ数が一致する("U-Net + MHSA, no positional
# encoding"に相当、EdgeAttNet論文のablationと同じ切り分け)。
# report/main.texが指摘してきた「edge priorそのものの効果か、単に
# attention容量が増えただけか」という交絡を分離するための唯一の差分。


class PlainMHSA(nn.Module):
    """edge priorなしの標準Multi-Head Self-Attention(bottleneck上)。

    EdgeGuidedMHSAと同じchannel数・head数・残差接続+LayerNormを保ちつつ、
    Q・Kへの edge_prior 加算だけを行わない(Q=K=V=bottleneck特徴量)。
    """

    def __init__(self, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        attn_out, _ = self.mhsa(x_flat, x_flat, x_flat)  # Q=K=V=x(edge priorなし)
        attn_out = self.norm(attn_out + x_flat)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegPlainMHSAUNet(nn.Module):
    """MediSegEdgeAttUNetからedge priorだけを除いた matched-parameter 対照群。

    入出力インターフェースはMediSegUNet/MediSegEdgeAttUNetと完全に同一。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.plain_attn = PlainMHSA(bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.plain_attn(x5)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


In [ ]:
# [4.5/8] 形状サニティチェック
_dummy = torch.randn(2, 3, CFG.CROP_SIZE, CFG.CROP_SIZE).cuda()
_probe = MediSegUNet(3, 1, False).cuda()
with torch.no_grad():
    _out = _probe(_dummy)
assert _out.shape == (2, 1, CFG.CROP_SIZE, CFG.CROP_SIZE), "出力形状が想定と異なる"
print({"n_params": sum(p.numel() for p in _probe.parameters())})
del _probe, _dummy, _out
torch.cuda.empty_cache()
print("形状チェックOK")


In [ ]:
# [5/8] Dataset・loss・学習/評価ループ
# baselineのrefiner_lossはV30から不変。dtb構成は境界項だけを距離変換版に置換する
# (region項・重み0.15・学習・評価ループは同一 = 単変数置換)
def square_box_from_bbox(bbox, height, width, context=CFG.CROP_CONTEXT, minimum=96):
    x, y, w, h = map(float, bbox)
    cx, cy = x + w / 2, y + h / 2
    side = int(math.ceil(max(w, h) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > width: x0 -= x1 - width; x1 = width
    if y1 > height: y0 -= y1 - height; y1 = height
    return max(0, x0), max(0, y0), min(width, x1), min(height, y1)

class InstanceCropDataset(Dataset):
    def __init__(self, file_names, all_labelers=True, augment=False):
        self.samples, self.augment = [], augment
        for file_name in file_names:
            for image_id in (file_to_ids[file_name] if all_labelers else []):
                for ann in annotations[image_id]:
                    if not ann.get("iscrowd", 0) and ann.get("area", 0) > 0:
                        self.samples.append((file_name, image_id, ann))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        file_name, image_id, ann = self.samples[index]
        info = images[image_id]
        image = cv2.imread(str(CFG.PREPROC_DIR / file_name), cv2.IMREAD_COLOR)
        target = ann_mask(image_id, ann)
        x0, y0, x1, y1 = square_box_from_bbox(ann["bbox"], info["height"], info["width"])
        image = cv2.resize(image[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_LINEAR)
        target = cv2.resize(target[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_NEAREST)
        image = torch.from_numpy(image.transpose(2, 0, 1).copy()).float() / 255.0
        target = torch.from_numpy(target[None].copy()).float()
        if self.augment:
            k = random.randrange(4)
            image, target = torch.rot90(image, k, (1, 2)), torch.rot90(target, k, (1, 2))
            if random.random() < 0.5:
                image, target = torch.flip(image, (2,)), torch.flip(target, (2,))
            if random.random() < 0.5:
                image, target = torch.flip(image, (1,)), torch.flip(target, (1,))
            if random.random() < 0.5:
                image[:2] = torch.clamp(image[:2] ** random.uniform(0.85, 1.20), 0, 1)
        return image, target

def soft_dice_loss(logits, targets):
    probs = logits.sigmoid()
    intersection = (probs * targets).sum((1, 2, 3))
    denominator = probs.sum((1, 2, 3)) + targets.sum((1, 2, 3))
    return (1 - (2 * intersection + 1) / (denominator + 1)).mean()

def boundary_map(tensor):
    dilation = F.max_pool2d(tensor, 5, stride=1, padding=2)
    erosion = -F.max_pool2d(-tensor, 5, stride=1, padding=2)
    return torch.clamp(dilation - erosion, 0, 1)

def refiner_loss(logits, targets):
    region = 0.35 * F.binary_cross_entropy_with_logits(logits, targets) + 0.65 * soft_dice_loss(logits, targets)
    edge = F.l1_loss(boundary_map(logits.sigmoid()), boundary_map(targets))
    return region + 0.15 * edge

def refiner_loss_dtb(logits, targets):
    """region項はrefiner_lossと同一。境界項を距離変換版 mean(sigmoid·φ_G) に置換。
    φ_G はaugment適用後のtargetから計算する(augmentと同期するためloss内で計算)。"""
    region = 0.35 * F.binary_cross_entropy_with_logits(logits, targets) + 0.65 * soft_dice_loss(logits, targets)
    phi_np = batch_signed_distance_maps(targets.detach().cpu().numpy())
    phi = torch.from_numpy(phi_np).to(logits.device)
    boundary = (logits.sigmoid()[:, 0] * phi).mean()
    return region + CFG.BDT_WEIGHT * boundary

def crop_dice_from_state(state_dict, loader, threshold=0.5):
    model = MediSegUNet(3, 1, False).cuda()
    model.load_state_dict(state_dict, strict=True)
    model.eval()
    total = count = 0
    with torch.no_grad():
        for image, target in loader:
            image, target = image.cuda(non_blocking=True), target.cuda(non_blocking=True).bool()
            pred = model(image).sigmoid() >= threshold
            intersection = (pred & target).sum((1, 2, 3)).float()
            denominator = pred.sum((1, 2, 3)) + target.sum((1, 2, 3))
            total += ((2 * intersection + 1) / (denominator + 1)).sum().item()
            count += len(image)
    del model
    torch.cuda.empty_cache()
    return total / max(count, 1)

def probe_batch_size(dataset, candidate, loss_fn):
    loader = DataLoader(dataset, batch_size=candidate, shuffle=True, num_workers=2, pin_memory=True)
    image = target = probe = loss = None
    try:
        image, target = next(iter(loader))
        probe = MediSegUNet(3, 1, False).cuda()
        with torch.amp.autocast("cuda"):
            loss = loss_fn(probe(image.cuda()), target.cuda())
        loss.backward()
    finally:
        del probe, image, target, loss, loader
        torch.cuda.empty_cache()

def choose_batch_size(dataset):
    for candidate in CFG.REFINER_BATCH_CANDIDATES:
        try:
            for loss_fn in (refiner_loss, refiner_loss_dtb):
                probe_batch_size(dataset, candidate, loss_fn)
            return candidate
        except torch.cuda.OutOfMemoryError:
            pass
    raise RuntimeError("No common batch size fits both paired losses")

def train_one_run(config, seed):
    checkpoint_path = CFG.ARTIFACT_ROOT / f"v59_bdt_pilot_{config}_seed{seed}_fold{CFG.FOLD}.pt"
    if checkpoint_path.exists():
        checkpoint = validate_checkpoint(torch.load(checkpoint_path, map_location="cpu", weights_only=False), config, seed)
        return {"checkpoint": str(checkpoint_path), **checkpoint["training_summary"]}
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    loss_fn = refiner_loss_dtb if config == "dtb" else refiner_loss
    train_ds = InstanceCropDataset(train_files_fold, augment=True)
    valid_ds = InstanceCropDataset(valid_files_fold, augment=False)
    batch_size = choose_batch_size(train_ds)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=4, pin_memory=True, persistent_workers=True)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False,
                              num_workers=4, pin_memory=True, persistent_workers=True)

    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = MediSegUNet(3, 1, False).cuda()
    initial = hashlib.sha256()
    for name, value in sorted(model.state_dict().items()):
        initial.update(name.encode()); initial.update(str(value.dtype).encode()); initial.update(str(tuple(value.shape)).encode())
        initial.update(value.detach().cpu().contiguous().numpy().tobytes())
    initial_state_sha256 = initial.hexdigest()
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.REFINER_LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.REFINER_EPOCHS)
    scaler = torch.amp.GradScaler("cuda")
    best_dice, best_state = -1.0, None
    started = time.time()

    for epoch in range(CFG.REFINER_EPOCHS):
        model.train()
        running = 0.0
        progress = tqdm(train_loader, desc=f"{config}/seed{seed} epoch {epoch + 1}/{CFG.REFINER_EPOCHS}")
        for image, target in progress:
            image, target = image.cuda(non_blocking=True), target.cuda(non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                loss = loss_fn(model(image), target)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running += loss.item()
            progress.set_postfix(loss=f"{running / (progress.n + 1):.4f}")
        scheduler.step()

        dice = crop_dice_from_state(model.state_dict(), valid_loader)
        if dice > best_dice:
            best_dice = dice
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"{config}/seed{seed} epoch {epoch + 1}: crop Dice {dice:.6f} (best {best_dice:.6f})")

    minutes = round((time.time() - started) / 60, 1)
    summary = {"best_crop_dice": best_dice, "minutes": minutes, "batch_size": batch_size,
               "training_minutes_scope": "original_complete_training_invocation"}
    atomic_save(checkpoint_path, {"model": model.state_dict(), "best_model": best_state, "best_dice": best_dice,
                "config": config, "seed": seed, "fold": CFG.FOLD, "input_contract": PILOT_CONTRACT,
                "training_status": "complete", "initial_state_sha256": initial_state_sha256,
                "training_summary": summary}, is_checkpoint=True)
    del model, optimizer, scheduler, train_loader, valid_loader, train_ds, valid_ds
    torch.cuda.empty_cache()
    return {"checkpoint": str(checkpoint_path), **summary}


In [ ]:
# [6/8] OOF採点用の後処理(V54/V51/matched-parameter ablationと同一: shrink@0.45 + drop_contained@0.90)
from ultralytics import YOLO
from pycocotools import mask as mask_utils

detector_path = CFG.DRIVE_ROOT / "runs" / f"yolo_fold_{CFG.FOLD}" / "weights" / "best.pt"
if file_sha256(detector_path) != PILOT_CONTRACT["detector_sha256"]:
    raise ValueError("Detector bytes changed before loading")
yolo_model = YOLO(str(detector_path))

def result_masks(result):
    if result.masks is None:
        return [], np.zeros(0, np.float32)
    height, width = result.orig_shape
    output = []
    for mask in result.masks.data.detach().cpu().numpy():
        if mask.shape != (height, width):
            mask = cv2.resize(mask, (width, height), interpolation=cv2.INTER_LINEAR)
        output.append(mask >= 0.5)
    confidences = result.boxes.conf.detach().cpu().numpy().astype(np.float32) if result.boxes is not None else np.zeros(0, np.float32)
    return output, confidences

def mask_square_box(mask, context=1.65, minimum=96):
    height, width = mask.shape
    ys, xs = np.where(mask)
    if not len(xs):
        return 0, 0, width, height
    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    side = int(math.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > width: x0 -= x1 - width; x1 = width
    if y1 > height: y0 -= y1 - height; y1 = height
    return max(0, x0), max(0, y0), min(width, x1), min(height, y1)

@torch.no_grad()
def refine_proposal(refiner, image, proposal):
    x0, y0, x1, y1 = mask_square_box(proposal)
    crop = cv2.resize(image[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_LINEAR)
    tensor = torch.from_numpy(crop.transpose(2, 0, 1)[None].copy()).float().cuda() / 255.0
    probability = refiner(tensor).sigmoid()[0, 0].cpu().numpy()
    probability = cv2.resize(probability, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR)
    refined = np.zeros_like(proposal, dtype=bool)
    refined[y0:y1, x0:x1] = probability >= CFG.REFINER_THRESHOLD
    return refined & proposal  # shrink(確定構成)

def greedy_disjoint(masks, order, min_area=CFG.MIN_AREA):
    occupied = np.zeros((2048, 2048), dtype=bool)
    kept = []
    for index in order:
        exclusive = masks[index] & ~occupied
        if exclusive.sum() < min_area:
            continue
        kept.append(exclusive)
        occupied |= exclusive
    return kept

def drop_contained_indices(masks, threshold=CFG.DROP_THRESHOLD):
    if len(masks) < 2:
        return list(range(len(masks)))
    rles = mask_utils.encode(np.asfortranarray(np.stack(masks, axis=-1).astype(np.uint8)))
    areas = np.asarray(mask_utils.area(rles), dtype=np.float64)
    self_iou = np.asarray(mask_utils.iou(rles, rles, [0] * len(rles)), dtype=np.float64)
    dropped = set()
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            if i in dropped or j in dropped:
                continue
            value = self_iou[i, j]
            if value <= 0:
                continue
            intersection = value * (areas[i] + areas[j]) / (1.0 + value)
            smaller = i if areas[i] <= areas[j] else j
            if intersection / max(areas[smaller], 1.0) >= threshold:
                dropped.add(smaller)
    return [i for i in range(len(masks)) if i not in dropped]

def apply_ownership(masks, scores):
    if not masks:
        return []
    areas = np.array([m.sum() for m in masks], dtype=np.float64)
    subset = drop_contained_indices(masks)
    return greedy_disjoint(masks, sorted(subset, key=lambda i: areas[i]))

def to_rle(annotation, height, width):
    segmentation = annotation["segmentation"]
    if isinstance(segmentation, list):
        return mask_utils.merge(mask_utils.frPyObjects(segmentation, height, width))
    if isinstance(segmentation.get("counts"), list):
        return mask_utils.frPyObjects(segmentation, height, width)
    rle = dict(segmentation)
    if isinstance(rle["counts"], str):
        rle["counts"] = rle["counts"].encode("utf-8")
    return rle

def iou_matrix(truth, pred):
    if not truth or not pred:
        return np.zeros((len(truth), len(pred)), np.float64)
    return np.asarray(mask_utils.iou(pred, truth, [0] * len(truth)), dtype=np.float64).T

class PanopticQuality:
    def __init__(self, iou_threshold=0.5):
        self.iou_threshold = iou_threshold
        self.tp_ious = []
        self.fp_count = 0
        self.fn_count = 0
    def add_matrix(self, matrix, n_gt, n_pred):
        if n_gt == 0:
            self.fp_count += n_pred; return
        if n_pred == 0:
            self.fn_count += n_gt; return
        hits = matrix > self.iou_threshold
        scores = matrix[hits].tolist()
        self.tp_ious.extend(scores)
        self.fp_count += int((hits.sum(axis=0) == 0).sum())
        self.fn_count += int((hits.sum(axis=1) == 0).sum())
    @property
    def denominator(self):
        return len(self.tp_ious) + 0.5 * self.fp_count + 0.5 * self.fn_count
    @property
    def score(self):
        d = self.denominator
        return float(sum(self.tp_ious) / d) if d > 0 else 0.0
    def sq_rq(self):
        tp = len(self.tp_ious)
        sq = float(sum(self.tp_ious) / tp) if tp > 0 else 0.0
        rq = float(tp / (tp + 0.5 * self.fp_count + 0.5 * self.fn_count)) if (tp + self.fp_count + self.fn_count) > 0 else 0.0
        return sq, rq
    def summary(self):
        sq, rq = self.sq_rq()
        return {"pq": self.score, "sq": sq, "rq": rq, "tp": len(self.tp_ious),
                "fp": self.fp_count, "fn": self.fn_count, "denominator": self.denominator}

gt_rles = {}
for image_id, info in images.items():
    height, width = int(info["height"]), int(info["width"])
    gt_rles[image_id] = [to_rle(ann, height, width) for ann in annotations[image_id] if not ann.get("iscrowd", 0)]

@torch.no_grad()
def evaluate_oof(refiner):
    refiner.eval()
    meter = PanopticQuality()
    for name in tqdm(valid_files_fold, desc=f"fold{CFG.FOLD} OOF採点"):
        image_path = CFG.PREPROC_DIR / name
        result = yolo_model.predict(str(image_path), imgsz=1280, conf=0.08, iou=0.55,
                                     agnostic_nms=True, retina_masks=True, verbose=False)[0]
        masks, confidences = result_masks(result)
        keep = [i for i, (m, s) in enumerate(zip(masks, confidences)) if s >= CFG.CONF and m.sum() >= CFG.MIN_AREA]
        proposals = [masks[i] for i in keep]
        scores = confidences[keep] if len(keep) else np.zeros(0, np.float32)
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        refined = [refine_proposal(refiner, image, m) for m in proposals]
        predictions = apply_ownership(refined, scores)
        pred_rles = []
        if predictions:
            encoded = mask_utils.encode(np.asfortranarray(np.stack(predictions, axis=-1).astype(np.uint8)))
            pred_rles = encoded if isinstance(encoded, list) else [encoded]
        for image_id in file_to_ids[name]:
            truth = gt_rles[image_id]
            matrix = iou_matrix(truth, pred_rles)
            meter.add_matrix(matrix, len(truth), len(pred_rles))
    return meter.summary()

print("後処理・採点関数の定義OK")


In [ ]:
# [7/8] Serial paired runs; legacy baselines are not implicitly relabelled.
manifest = read_pilot_manifest()
rows = manifest["runs"]
done_keys = {(r["config"], r["seed"]) for r in rows}
if CFG.REUSE_V54_BASELINE and CFG.V54_MANIFEST.exists():
    for row in reuse_baselines(json.loads(CFG.V54_MANIFEST.read_text(encoding="utf-8"))):
        if (row["config"], row["seed"]) not in done_keys:
            rows.append(row)
            done_keys.add((row["config"], row["seed"]))
manifest.pop("failure", None)
manifest["status"] = "in_progress"
try:
    for config in CFG.CONFIGS:
        for seed in CFG.SEEDS:
            if (config, seed) in done_keys:
                continue
            checkpoint_path = CFG.ARTIFACT_ROOT / f"v59_bdt_pilot_{config}_seed{seed}_fold{CFG.FOLD}.pt"
            scope = "evaluation_only" if checkpoint_path.exists() else "training_and_evaluation"
            started = time.time()
            torch.cuda.reset_peak_memory_stats()
            train_summary = train_one_run(config, seed)
            ckpt = validate_checkpoint(torch.load(train_summary["checkpoint"], map_location="cpu", weights_only=False), config, seed)
            refiner = MediSegUNet(3, 1, False).cuda().eval()
            refiner.load_state_dict(ckpt["best_model"], strict=True)
            if file_sha256(detector_path) != PILOT_CONTRACT["detector_sha256"]:
                raise ValueError("Detector bytes changed after input-contract construction")
            oof = evaluate_oof(refiner)
            row = {"config": config, "seed": seed, "fold": CFG.FOLD,
                   "bdt_weight": CFG.BDT_WEIGHT if config == "dtb" else None,
                   **train_summary, **oof, "input_contract": PILOT_CONTRACT,
                   "checkpoint_sha256": file_sha256(train_summary["checkpoint"]),
                   "initial_state_sha256": ckpt["initial_state_sha256"], "checkpoint_selection": "best_model",
                   "execution": {"service": "Google Colab", "project": str(CFG.DRIVE_ROOT),
                       "notebook": f"solar_filament_v59_boundary_dt_fold{CFG.FOLD}_colab.ipynb",
                       "gpu_name": torch.cuda.get_device_name(0),
                       "measurement_scope": scope, "duration_seconds": round(time.time() - started, 3),
                       "peak_vram_bytes": int(torch.cuda.max_memory_allocated()),
                       "completed_at_utc": datetime.now(timezone.utc).isoformat(),
                       "compute_units_used": None, "compute_units_note": "Unavailable from runtime API; record from Colab UI."}}
            validate_run_row(row)
            validate_paired_rows(rows + [row])
            rows.append(row)
            done_keys.add((config, seed))
            manifest["runs"] = rows
            manifest["updated_at_utc"] = datetime.now(timezone.utc).isoformat()
            atomic_save(CFG.MANIFEST, manifest)
            print(json.dumps(row, ensure_ascii=False, allow_nan=False))
            del refiner, ckpt
            torch.cuda.empty_cache()
except (Exception, KeyboardInterrupt) as error:
    manifest["status"] = "interrupted" if isinstance(error, KeyboardInterrupt) else "failed"
    try:
        failure_peak = int(torch.cuda.max_memory_allocated())
        failure_note = None
    except Exception as stats_error:
        failure_peak = None
        failure_note = f"CUDA memory statistics unavailable: {type(stats_error).__name__}"
    manifest["failure"] = {"type": type(error).__name__, "at_utc": datetime.now(timezone.utc).isoformat(),
        "service": "Google Colab", "project": str(CFG.DRIVE_ROOT),
        "notebook": f"solar_filament_v59_boundary_dt_fold{CFG.FOLD}_colab.ipynb",
        "gpu_name": torch.cuda.get_device_name(0), "measurement_scope": scope,
        "duration_seconds": round(time.time() - started, 3), "peak_vram_bytes": failure_peak,
        "peak_vram_unavailable_reason": failure_note, "compute_units_used": None,
        "compute_units_note": "Unavailable from runtime API; record from Colab UI."}
    atomic_save(CFG.MANIFEST, manifest)
    raise
manifest["status"] = "complete"
atomic_save(CFG.MANIFEST, manifest)

# --- 集計と採否の目安 ---
import statistics
pair = {}
for row in rows:
    pair.setdefault(row["seed"], {})[row["config"]] = row
print("\n=== seed別 paired Δ(dtb - baseline) ===")
deltas = []
for seed, arms in sorted(pair.items()):
    if "baseline" in arms and "dtb" in arms:
        delta = arms["dtb"]["pq"] - arms["baseline"]["pq"]
        deltas.append(delta)
        print(f"seed {seed}: ΔPQ {delta:+.6f} (dtb {arms['dtb']['pq']:.6f} vs baseline {arms['baseline']['pq']:.6f})")
if deltas:
    print(f"\nmean Δ {statistics.mean(deltas):+.6f} / 正のseed {sum(d > 0 for d in deltas)}/{len(deltas)}")
print("\n採否の目安: 2seedで符号一致して初めてfold1・2へ独立確認。境界項の効果は"
      "SQ(分割質)に現れるはずなので、PQが動かなくてもSQの系統的改善があれば次の手を検討。")
